# CSE5280 — Robot Arm Crowd Interference Simulation
**Author:** Michael Caruso  |  **Course:** CSE5280  |  **Spring 2026**

---

## Table of Contents
1. Environment Setup
2. Mathematical Framework
3. Building & Obstacles
4. Cost Functions
5. Agent Initialization
6. Optimizer
7. Robot Arm (IK)
8. Cluster Tracker
9. Visualization
10. Simulation Loop
11. Experiments
12. Discussion

---

### Overview

Single flat room with two exits on the south wall and two interior pillar obstacles.
A 3-DOF arm is mounted **at the centre** of the room so it can intercept both exits.
The robot tracks each exit independently, locks onto whichever stream is larger,
and drives its end-effector there — forcing agents to switch exits repeatedly.


In [ ]:
#@title # 1. Environment Setup { display-mode: "form" }
import os
os.system("pip install -q vedo")
os.system("apt-get install -q -y xvfb freeglut3-dev ffmpeg > /dev/null 2>&1")
os.system("Xvfb :99 -screen 0 1920x1080x24 &")
os.environ["DISPLAY"] = ":99"

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import vedo
from vedo import (Sphere, Box, Cylinder, Plotter, Video,
                  Light, settings as vedo_settings)
from sklearn.cluster import KMeans
from IPython.display import Image as IPImage, display as ipy_display

vedo_settings.default_backend = "vtk"
print("Environment ready.  vedo:", vedo.__version__)


## 2. Mathematical Framework

### Total Agent Cost

$$C = C_{\text{goal}} + C_{\text{walls}} + C_{\text{obstacles}} + C_{\text{repulsion}} + C_{\text{robot}} + C_{\text{smooth}}$$

**Soft-min exit:** $C_{\text{goal}} = -\tau\log\sum_i e^{-\|\mathbf{p}-\mathbf{e}_i\|^2/\tau}$

**Analytical wall/obstacle gradient** (closest-point-on-segment):
$\nabla C_w = \sum_k e^{-d_k^2/2\sigma_w^2}\,(-\Delta_k/\sigma_w^2)$

**Vectorised repulsion** ($O(K^2)$ NumPy):
$\nabla C_{r,i} = \sum_{j\neq i} e^{-\|\Delta_{ij}\|^2/2\sigma_r^2}\,(\Delta_{ij}/\sigma_r^2)$

**Robot EE obstacle:**
$C_{\text{robot}} = w_r\,e^{-\|\mathbf{p}-\mathbf{e}(t)\|^2/2\sigma_r^2}$

**Prediction:** $\hat{\mathbf{p}} = \boldsymbol{\mu}^{(t)} + T_h\Delta\boldsymbol{\mu}$

**Jacobian IK:** $\boldsymbol{\phi}^{k+1} = \boldsymbol{\phi}^k + J^+\lambda(\hat{\mathbf{p}}-\mathbf{e}^k)$


In [ ]:
#@title # 3. Building & Obstacles { display-mode: "form" }
#@markdown Flat box with two exits and two interior pillar obstacles.
#@markdown Obstacles force agents to navigate around the robot base
#@markdown and create more dynamic, realistic evacuation streams.

# ── Room dimensions ──────────────────────────────────────────────────────────
FLOOR_W = 15.0   # half-width  x
FLOOR_D = 12.0   # half-depth  y
Z_FLOOR = 0.0
WALL_H  = 4.5

# ── Exits (south wall) ────────────────────────────────────────────────────────
EXIT_GAP  = 2.2
EXIT_A    = np.array([ 7.5, -FLOOR_D, Z_FLOOR])   # right exit
EXIT_B    = np.array([-7.5, -FLOOR_D, Z_FLOOR])   # left  exit
EXITS     = [EXIT_A, EXIT_B]
EXIT_THRESH = 1.6

_EAx, _EBx = EXIT_A[0], EXIT_B[0]

# ── Interior pillar obstacles (x_centre, y_centre, half-size) ────────────────
# Two pillars that channel agents into streams — makes the robot's job harder
PILLARS = [
    {"cx":  5.5, "cy": -4.0, "hw": 1.2, "hd": 1.2},   # right-centre
    {"cx": -5.5, "cy": -4.0, "hw": 1.2, "hd": 1.2},   # left-centre
]

# ── 2-D wall + obstacle segments for analytical gradient ──────────────────────
WALL_SEGS_2D = [
    # Boundary
    (np.array([-FLOOR_W,  FLOOR_D]), np.array([ FLOOR_W,  FLOOR_D])),  # N
    (np.array([ FLOOR_W,  FLOOR_D]), np.array([ FLOOR_W, -FLOOR_D])),  # E
    (np.array([-FLOOR_W,  FLOOR_D]), np.array([-FLOOR_W, -FLOOR_D])),  # W
    # South wall — three segments with exit gaps
    (np.array([-FLOOR_W,        -FLOOR_D]), np.array([_EBx-EXIT_GAP, -FLOOR_D])),
    (np.array([_EBx+EXIT_GAP,   -FLOOR_D]), np.array([_EAx-EXIT_GAP, -FLOOR_D])),
    (np.array([_EAx+EXIT_GAP,   -FLOOR_D]), np.array([ FLOOR_W,      -FLOOR_D])),
]

# Pillar edge segments (4 sides each)
for p in PILLARS:
    cx, cy, hw, hd = p["cx"], p["cy"], p["hw"], p["hd"]
    WALL_SEGS_2D += [
        (np.array([cx-hw, cy-hd]), np.array([cx+hw, cy-hd])),  # south
        (np.array([cx+hw, cy-hd]), np.array([cx+hw, cy+hd])),  # east
        (np.array([cx+hw, cy+hd]), np.array([cx-hw, cy+hd])),  # north
        (np.array([cx-hw, cy+hd]), np.array([cx-hw, cy-hd])),  # west
    ]

print(f"Room: {2*FLOOR_W:.0f} x {2*FLOOR_D:.0f}  |  {len(PILLARS)} pillars  "
      f"|  {len(WALL_SEGS_2D)} wall/obstacle segments")
print(f"Exit A: {EXIT_A}   Exit B: {EXIT_B}")


In [ ]:
#@title # 4. Cost Functions { display-mode: "form" }
#@markdown All analytical gradients — no finite differences anywhere.

TAU         = 1.5    # soft-min temperature
W_GOAL      = 1.0
W_WALL      = 6.0    # boundary + pillar repulsion
SIGMA_WALL  = 1.4
W_REPULSION = 2.0
SIGMA_REP   = 1.3
W_SMOOTH    = 0.04
W_ROBOT     = 14.0   # strong robot obstacle — must visibly scatter agents
SIGMA_ROBOT = 3.0    # influence radius (building units)


def grad_goal_softmin(pos):
    costs    = np.array([np.sum((pos - e)**2) for e in EXITS])
    c_min    = costs.min()
    exp_vals = np.exp(-(costs - c_min) / TAU)
    weights  = exp_vals / exp_vals.sum()
    grad     = np.zeros(3)
    for i, e in enumerate(EXITS):
        grad += weights[i] * 2.0 * (pos - e)
    return W_GOAL * grad


def grad_walls(pos):
    """Repulsion from all wall + pillar segments (analytical)."""
    p2d = pos[:2]
    g2d = np.zeros(2)
    for a, b in WALL_SEGS_2D:
        v     = b - a
        denom = np.dot(v, v)
        t     = (np.clip(np.dot(p2d - a, v) / denom, 0.0, 1.0)
                 if denom > 1e-12 else 0.0)
        diff  = p2d - (a + t * v)
        d     = np.linalg.norm(diff)
        if d < 1e-8:
            continue
        g2d += np.exp(-d**2 / (2*SIGMA_WALL**2)) * (-diff / SIGMA_WALL**2)
    grad     = np.zeros(3)
    grad[:2] = W_WALL * g2d
    return grad


def compute_all_repulsions(positions, active_mask):
    N     = len(positions)
    grads = np.zeros((N, 3))
    idx   = np.where(active_mask)[0]
    if len(idx) < 2:
        return grads
    p2d  = positions[idx, :2]
    diff = p2d[:, None, :] - p2d[None, :, :]      # (K,K,2)
    d_sq = np.sum(diff**2, axis=2)                 # (K,K)
    np.fill_diagonal(d_sq, 1e10)
    w    = np.where(d_sq >= 1e-4,
                    np.exp(-d_sq / (2*SIGMA_REP**2)), 0.0)
    g2d  = np.sum(w[:, :, None] * diff / SIGMA_REP**2, axis=1)
    grads[idx, :2] = W_REPULSION * g2d
    return grads


def grad_robot_obstacle(pos, ee_pos3d):
    diff = pos - ee_pos3d
    d2   = np.dot(diff, diff)
    return W_ROBOT * np.exp(-d2 / (2*SIGMA_ROBOT**2)) * (-diff / SIGMA_ROBOT**2)


print("Cost functions OK.")
_g = grad_goal_softmin(np.array([0., 0., 0.]))
print(f"  Goal grad at origin: y={_g[1]:.3f}  (negative = south, toward exits ✓)")


In [ ]:
#@title # 5. Agent Initialization { display-mode: "form" }
#@markdown Agents placed with minimum spacing, away from exits and pillars.

def _inside_pillar(xy):
    """True if xy is inside any pillar (with a 0.5-unit buffer)."""
    for p in PILLARS:
        if (abs(xy[0] - p["cx"]) < p["hw"] + 0.5 and
                abs(xy[1] - p["cy"]) < p["hd"] + 0.5):
            return True
    return False


def initialize_agents(n_agents=40, seed=42):
    rng      = np.random.default_rng(seed)
    MIN_DIST = 1.6

    positions = []
    attempts  = 0
    while len(positions) < n_agents and attempts < 100000:
        attempts += 1
        x = rng.uniform(-FLOOR_W + 2.0,  FLOOR_W - 2.0)
        y = rng.uniform(-FLOOR_D + 5.0,  FLOOR_D - 2.5)  # no spawning near exits
        xy = np.array([x, y])
        if _inside_pillar(xy):
            continue
        candidate = np.array([x, y, Z_FLOOR])
        if positions:
            dists = np.linalg.norm(
                np.array(positions)[:, :2] - xy, axis=1)
            if dists.min() < MIN_DIST:
                continue
        positions.append(candidate)

    if len(positions) < n_agents:
        print(f"  Warning: placed {len(positions)}/{n_agents}")

    positions  = np.array(positions)
    velocities = np.zeros_like(positions)
    prev_pos   = positions.copy()
    active     = np.ones(len(positions), dtype=bool)
    return positions, velocities, prev_pos, active


pos_t, _, _, _ = initialize_agents(40)
print(f"Placed {len(pos_t)} agents")
print(f"  x ∈ [{pos_t[:,0].min():.1f}, {pos_t[:,0].max():.1f}]")
print(f"  y ∈ [{pos_t[:,1].min():.1f}, {pos_t[:,1].max():.1f}]")


In [ ]:
#@title # 6. Gradient Descent Optimizer { display-mode: "form" }

ALPHA    = 0.045
BETA     = 0.72     # high momentum → smooth flow
MAX_STEP = 0.22


def step_agents(positions, velocities, prev_pos, active, ee_pos3d):
    new_pos  = positions.copy()
    new_vel  = velocities.copy()
    new_prev = prev_pos.copy()
    new_act  = active.copy()
    all_rep  = compute_all_repulsions(positions, active)

    for i in range(len(positions)):
        if not active[i]:
            continue
        for ex in EXITS:
            if np.linalg.norm(positions[i] - ex) < EXIT_THRESH:
                new_act[i] = False
                break
        if not new_act[i]:
            continue

        pos = positions[i]
        g   = (grad_goal_softmin(pos)
               + grad_walls(pos)
               + all_rep[i]
               + grad_robot_obstacle(pos, ee_pos3d)
               + W_SMOOTH * 2.0 * (pos - prev_pos[i]))

        v_new     = BETA * velocities[i] - ALPHA * g
        step_norm = np.linalg.norm(v_new)
        if step_norm > MAX_STEP:
            v_new *= MAX_STEP / step_norm

        new_p    = positions[i] + v_new
        new_p[0] = np.clip(new_p[0], -FLOOR_W + 0.3,  FLOOR_W - 0.3)
        new_p[1] = np.clip(new_p[1], -FLOOR_D - 1.0,  FLOOR_D - 0.3)
        new_p[2] = Z_FLOOR

        new_prev[i] = positions[i]
        new_pos[i]  = new_p
        new_vel[i]  = v_new

    return new_pos, new_vel, new_prev, new_act


print(f"Optimizer: α={ALPHA}, β={BETA}, max_step={MAX_STEP}")


In [ ]:
#@title # 7. Robot Arm — Inverse Kinematics { display-mode: "form" }
#@markdown 3-DOF arm mounted at the CENTRE of the room.
#@markdown This gives equal reach to both exits and makes every interference
#@markdown attempt visually dramatic — the arm sweeps across the whole floor.

L1 = 3.0    # pedestal height
L2 = 6.5    # upper arm
L3 = 9.0    # forearm  (L2+L3 = 15.5 > 14.15 = dist to exits ✓)

# ── Centre mount ─────────────────────────────────────────────────────────────
BASE_POS = np.array([0.0, 1.0, Z_FLOOR])   # slightly north of centre


class RobotArm:
    """
    3-DOF arm: base yaw (z) + shoulder pitch (y) + elbow pitch (y).
    phi = [phi0_deg, phi1_deg, phi2_deg]
    """

    def __init__(self, base_pos, l1=L1, l2=L2, l3=L3):
        self.base  = np.array(base_pos, dtype=float)
        self.l1, self.l2, self.l3 = l1, l2, l3
        self.dphi  = 0.5    # Jacobian step (degrees)
        self.lam   = 0.05   # IK gain

    @staticmethod
    def _Rz(d):
        c, s = np.cos(np.radians(d)), np.sin(np.radians(d))
        return np.array([[c,-s,0],[s,c,0],[0,0,1]])

    @staticmethod
    def _Ry(d):
        c, s = np.cos(np.radians(d)), np.sin(np.radians(d))
        return np.array([[c,0,s],[0,1,0],[-s,0,c]])

    def forward_kinematics(self, phi):
        p0 = self.base.copy()
        p1 = p0 + np.array([0., 0., self.l1])

        Rz0    = self._Rz(phi[0])
        Ry1    = self._Ry(phi[1])
        upper  = Rz0 @ Ry1 @ np.array([0., 0., 1.])
        p2     = p1 + self.l2 * upper

        Ry2    = self._Ry(phi[2])
        fore   = Rz0 @ Ry1 @ Ry2 @ np.array([0., 0., 1.])
        p3     = p2 + self.l3 * fore
        return p0, p1, p2, p3

    def get_endeffector_pos(self, phi):
        return self.forward_kinematics(phi)[3]

    def jacobian_matrix(self, phi):
        e    = self.get_endeffector_pos(phi)
        cols = []
        for i in range(3):
            dp = np.zeros(3); dp[i] = self.dphi
            ep = self.get_endeffector_pos(phi + dp)
            cols.append((ep - e) / np.radians(self.dphi))
        return np.column_stack(cols)

    def ik_step(self, phi, target, lam=None):
        if lam is None:
            lam = self.lam
        e    = self.get_endeffector_pos(phi)
        J    = self.jacobian_matrix(phi)
        dphi = np.linalg.pinv(J) @ (lam * (np.array(target) - e))
        return phi + np.degrees(dphi)


robot    = RobotArm(BASE_POS)
phi_init = np.array([0., -30., 60.])  # arm resting upward, slightly tilted

# Sanity-check IK on both exits
for name, ex in [("EXIT_A", EXIT_A), ("EXIT_B", EXIT_B)]:
    phi_t = phi_init.copy()
    for _ in range(800):
        phi_t = robot.ik_step(phi_t, ex, lam=0.06)
    ee    = robot.get_endeffector_pos(phi_t)
    print(f"IK → {name}: EE={ee.round(2)}  err={np.linalg.norm(ee-ex):.3f}")


# ── Rendering helpers ─────────────────────────────────────────────────────────
_C_BASE  = [30,  55, 160]
_C_UPPER = [55,  95, 200]
_C_FORE  = [80, 145, 225]
_C_JOINT = [210, 210, 220]
_C_EE    = [ 40, 220,  90]

_RL = 0.28    # link cylinder radius
_RJ = 0.40    # joint sphere radius
_RE = 0.55    # end-effector sphere radius


def _cyl(a, b, r, c):
    a, b = np.array(a, float), np.array(b, float)
    h = np.linalg.norm(b - a)
    if h < 1e-6:
        return None
    obj = Cylinder(pos=(a+b)/2, r=r, height=h, axis=b-a, c=c)
    return obj.lighting("default", ambient=0.4, diffuse=0.7, specular=0.3)


def _sph(pos, r, c):
    return (Sphere(pos=pos, r=r, c=c)
            .lighting("default", ambient=0.4, diffuse=0.7, specular=0.4))


def make_arm_meshes(phi):
    p0, p1, p2, p3 = robot.forward_kinematics(phi)
    objs = []
    c = _cyl(p0, p1, _RL*1.5, _C_BASE);  objs.append(c) if c else None
    objs.append(_sph(p1, _RJ*1.3, _C_JOINT))
    c = _cyl(p1, p2, _RL, _C_UPPER);     objs.append(c) if c else None
    objs.append(_sph(p2, _RJ, _C_JOINT))
    c = _cyl(p2, p3, _RL*0.8, _C_FORE);  objs.append(c) if c else None
    objs.append(_sph(p3, _RE, _C_EE))
    return [o for o in objs if o is not None]


print("Robot arm ready.  Base:", BASE_POS)
print(f"  Links: L1={L1}, L2={L2}, L3={L3}  reach≈{L2+L3:.1f}")


In [ ]:
#@title # 8. Cluster Tracker — Dual-Exit { display-mode: "form" }
#@markdown Tracks a cluster near EACH exit independently.
#@markdown Robot targets whichever exit has the LARGER active cluster.
#@markdown This ensures both exits are threatened, creating visible switching.

CLUSTER_EXIT_RADIUS = 7.0
PREDICT_HORIZON     = 5


class ExitCluster:
    """Tracks one cluster centred near a single exit."""

    def __init__(self, exit_pos):
        self.exit_pos      = np.array(exit_pos)
        self.centroid      = None
        self.prev_centroid = None
        self.velocity      = np.zeros(3)
        self.size          = 0

    def update(self, positions, active):
        active_pos = positions[active]
        near = [p for p in active_pos
                if np.linalg.norm(p[:2] - self.exit_pos[:2]) < CLUSTER_EXIT_RADIUS]
        self.size = len(near)
        if self.size < 2:
            self.centroid = None
            return
        c3d = np.mean(near, axis=0)
        self.prev_centroid = self.centroid
        self.centroid      = c3d
        self.velocity      = (c3d - self.prev_centroid
                               if self.prev_centroid is not None
                               else np.zeros(3))

    def predict(self, horizon=PREDICT_HORIZON):
        if self.centroid is None:
            return None
        pred    = self.centroid + horizon * self.velocity
        pred[0] = np.clip(pred[0], -FLOOR_W+1, FLOOR_W-1)
        pred[1] = np.clip(pred[1], -FLOOR_D+0.5, self.exit_pos[1])
        pred[2] = Z_FLOOR
        return pred


cluster_A = ExitCluster(EXIT_A)
cluster_B = ExitCluster(EXIT_B)


def pick_robot_target(positions, active):
    """
    Update both exit clusters and return the predicted intercept
    for whichever cluster is currently larger.
    Returns (target_3d, dominant_exit_index, cluster_A, cluster_B).
    """
    cluster_A.update(positions, active)
    cluster_B.update(positions, active)

    # Choose larger cluster
    if cluster_A.size >= cluster_B.size and cluster_A.centroid is not None:
        tgt  = cluster_A.predict()
        dominant = 0
    elif cluster_B.centroid is not None:
        tgt  = cluster_B.predict()
        dominant = 1
    else:
        tgt, dominant = None, -1
    return tgt, dominant


print("Dual-exit cluster tracker ready.")
print(f"  Tracking radius: {CLUSTER_EXIT_RADIUS} units  |  "
      f"Prediction horizon: {PREDICT_HORIZON} steps")


In [ ]:
#@title # 9. Visualization { display-mode: "form" }

CLR_FLOOR     = [245, 245, 240]
CLR_WALL      = [185, 185, 185]
CLR_PILLAR    = [155, 120,  80]   # warm tan — pillars
CLR_EXIT_A    = [ 50, 200,  80]   # green
CLR_EXIT_B    = [ 45, 125, 235]   # blue
CLR_AGENT_A   = [240,  95,  35]   # orange — heading exit A
CLR_AGENT_B   = [ 35, 125, 240]   # blue   — heading exit B
CLR_EVACUATED = [155, 155, 155]
CLR_CLUSTER_A = [255, 200,   0]   # yellow — cluster A centroid
CLR_CLUSTER_B = [  0, 210, 210]   # cyan   — cluster B centroid
CLR_TARGET    = [255,  45,  45]   # red    — predicted intercept

AGENT_R = 0.42
HIDDEN  = np.array([999., 999., 999.])


def matte(mesh, alpha=1.0):
    return mesh.lighting(
        "default", ambient=0.45, diffuse=0.65, specular=0.05).alpha(alpha)


def build_static_scene():
    objs = []
    t = 0.35
    # Floor
    objs.append(matte(Box(pos=(0,0,-0.1),
                          length=2*FLOOR_W, width=2*FLOOR_D,
                          height=0.2, c=CLR_FLOOR)))
    # Boundary walls
    for pos, lx, ly in [
        ((0,  FLOOR_D, WALL_H/2), 2*FLOOR_W, t),
        (( FLOOR_W, 0, WALL_H/2), t, 2*FLOOR_D),
        ((-FLOOR_W, 0, WALL_H/2), t, 2*FLOOR_D),
    ]:
        objs.append(matte(Box(pos=pos, length=lx, width=ly,
                              height=WALL_H, c=CLR_WALL), 0.68))
    # South wall segments
    segs_x = [
        (-FLOOR_W,           _EBx-EXIT_GAP),
        (_EBx+EXIT_GAP,      _EAx-EXIT_GAP),
        (_EAx+EXIT_GAP,      FLOOR_W),
    ]
    for x0, x1 in segs_x:
        w = abs(x1 - x0)
        if w < 0.05: continue
        objs.append(matte(Box(pos=((x0+x1)/2, -FLOOR_D, WALL_H/2),
                              length=w, width=t,
                              height=WALL_H, c=CLR_WALL), 0.68))
    # Exit markers
    for ex, clr in zip(EXITS, [CLR_EXIT_A, CLR_EXIT_B]):
        objs.append(matte(Box(pos=(ex[0], ex[1]+0.05, 0.07),
                              length=EXIT_GAP*2, width=0.25,
                              height=0.14, c=clr)))
        objs.append(matte(Box(pos=(ex[0], ex[1]+0.2, WALL_H*0.5),
                              length=EXIT_GAP*2, width=0.1,
                              height=WALL_H, c=clr), 0.22))
    # Interior pillars
    for p in PILLARS:
        objs.append(matte(Box(
            pos=(p["cx"], p["cy"], WALL_H*0.4),
            length=p["hw"]*2, width=p["hd"]*2,
            height=WALL_H*0.8, c=CLR_PILLAR)))
    return objs


def make_agent_spheres(positions, active):
    return [matte(Sphere(pos=pos, r=AGENT_R,
                         c=(CLR_EVACUATED if not act
                            else CLR_AGENT_A if pos[0] >= 0
                            else CLR_AGENT_B)))
            for pos, act in zip(positions, active)]


def agent_color(pos, act):
    if not act:
        return CLR_EVACUATED
    costs   = np.array([np.sum((pos - e)**2) for e in EXITS])
    c_min   = costs.min()
    w       = np.exp(-(costs - c_min) / TAU)
    w      /= w.sum()
    if w[0] > 0.55: return CLR_AGENT_A
    if w[1] > 0.55: return CLR_AGENT_B
    return [150, 70, 210]   # purple = undecided


# ── Preview ───────────────────────────────────────────────────────────────────
static_objs = build_static_scene()
agent_sph   = make_agent_spheres(pos_t, np.ones(len(pos_t), dtype=bool))
arm_prev    = make_arm_meshes(phi_init)
dir_light   = Light(pos=(18,-28,50), focal_point=(0,-2,1), intensity=0.9)

plt_prev = Plotter(title="Preview", size=(1280,720),
                   bg="white", bg2="whitesmoke",
                   interactive=False, offscreen=True)
plt_prev.show(*static_objs, *agent_sph, *arm_prev, dir_light,
              viewup="z",
              camera={"pos":(40,-55,42), "focalPoint":(0,-2,1)})
plt_prev.screenshot("/content/building_preview.png")
plt_prev.close()
ipy_display(IPImage("/content/building_preview.png"))
print("Preview rendered.")


In [ ]:
#@title # 10. Simulation Loop { display-mode: "form" }

N_AGENTS_SIM       = 40   #@param {type:"integer"}
MAX_STEPS_SIM      = 400  #@param {type:"integer"}
VIDEO_FPS          = 15   #@param {type:"integer"}
IK_STEPS_PER_FRAME = 8    #@param {type:"integer"}
VIDEO_FILE         = "/content/robot_evacuation.mp4"

vedo_settings.default_backend = "vtk"

# ── Init ─────────────────────────────────────────────────────────────────────
positions, velocities, prev_pos, active = initialize_agents(N_AGENTS_SIM, seed=42)
evac_times = [None] * N_AGENTS_SIM
cluster_A  = ExitCluster(EXIT_A)
cluster_B  = ExitCluster(EXIT_B)

phi       = phi_init.copy()
ee_pos3d  = robot.get_endeffector_pos(phi).copy()
robot_tgt = np.array([0., -FLOOR_D + 3., 0.])  # initial aim: south centre

# ── Build scene ──────────────────────────────────────────────────────────────
static_objs   = build_static_scene()
agent_spheres = make_agent_spheres(positions, active)
arm_meshes    = make_arm_meshes(phi)

# Persistent cluster + target indicators
HIDDEN     = np.array([999., 999., 999.])
ee_halo    = Sphere(pos=ee_pos3d, r=SIGMA_ROBOT, c=[60,230,100], alpha=0.08)
csph_A     = matte(Sphere(pos=HIDDEN, r=0.65, c=CLR_CLUSTER_A))
csph_B     = matte(Sphere(pos=HIDDEN, r=0.65, c=CLR_CLUSTER_B))
tgt_sph    = matte(Sphere(pos=HIDDEN, r=0.70, c=CLR_TARGET))
dir_light  = Light(pos=(18,-28,50), focal_point=(0,-2,1), intensity=0.9)

plt_sim = Plotter(
    title=f"Robot Arm Evacuation  {N_AGENTS_SIM} Agents",
    size=(1280, 720), bg="white", bg2="whitesmoke",
    interactive=False, offscreen=True)
camera = dict(pos=(40,-55,42), focal_point=(0,-2,1), viewup=(0,0,1))
plt_sim.show(*static_objs, *agent_spheres, *arm_meshes,
             ee_halo, csph_A, csph_B, tgt_sph,
             dir_light, camera=camera)

video = Video(VIDEO_FILE, fps=VIDEO_FPS, backend="ffmpeg")
video.add_frame()
print(f"Simulation: {N_AGENTS_SIM} agents, {MAX_STEPS_SIM} steps")

prev_dominant = -1   # track which exit the robot targeted last

for step in range(MAX_STEPS_SIM):
    n_active = int(active.sum())
    if n_active == 0:
        print(f"  All evacuated at step {step}.")
        break

    # 1. Cluster detection — both exits
    cluster_A.update(positions, active)
    cluster_B.update(positions, active)

    csph_A.pos(cluster_A.centroid if cluster_A.centroid is not None else HIDDEN)
    csph_B.pos(cluster_B.centroid if cluster_B.centroid is not None else HIDDEN)

    # Pick dominant cluster
    if cluster_A.size >= cluster_B.size and cluster_A.centroid is not None:
        pred = cluster_A.predict()
        dominant = 0
    elif cluster_B.centroid is not None:
        pred = cluster_B.predict()
        dominant = 1
    else:
        pred, dominant = None, prev_dominant

    if pred is not None:
        robot_tgt = pred.copy()
        tgt_sph.pos(pred)
        if dominant != prev_dominant and prev_dominant != -1:
            print(f"  Step {step:4d}: Robot switches → "
                  f"Exit {"A" if dominant==0 else "B"} "
                  f"(sizes A={cluster_A.size} B={cluster_B.size})")
    else:
        tgt_sph.pos(HIDDEN)
    prev_dominant = dominant

    # 2. IK
    for _ in range(IK_STEPS_PER_FRAME):
        phi = robot.ik_step(phi, robot_tgt)
    ee_pos3d = robot.get_endeffector_pos(phi).copy()
    ee_halo.pos(ee_pos3d)

    # 3. Agent step
    prev_active = active.copy()
    positions, velocities, prev_pos, active = step_agents(
        positions, velocities, prev_pos, active, ee_pos3d)
    for i in range(N_AGENTS_SIM):
        if prev_active[i] and not active[i]:
            evac_times[i] = step

    # 4. Update render
    for sphere, pos, act in zip(agent_spheres, positions, active):
        sphere.pos(pos)
        sphere.color(agent_color(pos, act)).alpha(0.45 if not act else 1.0)

    for m in arm_meshes:
        plt_sim.remove(m)
    arm_meshes = make_arm_meshes(phi)
    plt_sim.add(*arm_meshes)

    plt_sim.render()
    video.add_frame()

    if step % 50 == 0:
        nA = sum(positions[i,0]>=0 and active[i] for i in range(N_AGENTS_SIM))
        nB = sum(positions[i,0]< 0 and active[i] for i in range(N_AGENTS_SIM))
        print(f"  Step {step:4d} | Active:{n_active:3d} "
              f"| →A:{int(nA):2d} →B:{int(nB):2d} "
              f"| ClustA:{cluster_A.size} ClustB:{cluster_B.size} "
              f"| EE=({ee_pos3d[0]:.1f},{ee_pos3d[1]:.1f},{ee_pos3d[2]:.1f})")

video.close()
plt_sim.close()
print(f"Done. Video: {VIDEO_FILE}")
print("Download from Files panel (folder icon, left sidebar).")


In [ ]:
#@title # 11. Experiments & Analysis { display-mode: "form" }

def run_exp(n_agents, max_steps=350, robot_active=True,
            horizon=PREDICT_HORIZON, seed=42):
    pos, vel, prev_p, act = initialize_agents(n_agents, seed=seed)
    evac = [None]*n_agents
    cA   = ExitCluster(EXIT_A)
    cB   = ExitCluster(EXIT_B)
    phi_e = phi_init.copy()
    ee_p  = robot.get_endeffector_pos(phi_e).copy()
    tgt   = np.array([0., -FLOOR_D+3., 0.])

    for step in range(max_steps):
        if not act.any(): break
        if robot_active:
            cA.update(pos, act); cB.update(pos, act)
            if cA.size >= cB.size and cA.centroid is not None:
                pred = cA.predict(horizon)
            elif cB.centroid is not None:
                pred = cB.predict(horizon)
            else:
                pred = None
            if pred is not None: tgt = pred.copy()
            for _ in range(IK_STEPS_PER_FRAME):
                phi_e = robot.ik_step(phi_e, tgt)
            ee_p = robot.get_endeffector_pos(phi_e).copy()
        else:
            ee_p = HIDDEN.copy()
        pos, vel, prev_p, act = step_agents(pos, vel, prev_p, act, ee_p)
        for i in range(n_agents):
            if not act[i] and evac[i] is None: evac[i] = step
    return evac


def evac_curve(times, n, max_steps=350):
    f = np.zeros(max_steps)
    for t in times:
        if t is not None: f[t:] += 1
    return np.arange(max_steps), f/n*100


print("Running experiments...")
ev_on   = run_exp(40, robot_active=True,  seed=42)
ev_off  = run_exp(40, robot_active=False, seed=42)
print("  A done")
ev_h0   = run_exp(35, horizon=0,  seed=1)
ev_h5   = run_exp(35, horizon=5,  seed=1)
ev_h12  = run_exp(35, horizon=12, seed=1)
print("  B done")
ev_20   = run_exp(20, seed=5)
ev_40   = run_exp(40, seed=5)
ev_70   = run_exp(70, seed=5)
print("  C done")

fig, axes = plt.subplots(1, 3, figsize=(16,5))
fig.suptitle("Robot Interference Experiments (centre mount, pillar obstacles)",
             fontsize=12, fontweight="bold")

ax = axes[0]
for ev, n, lbl, c in [(ev_on,40,"Robot ON","#D03535"),
                       (ev_off,40,"Robot OFF","#3570D0")]:
    s,f = evac_curve(ev,n); ax.plot(s,f,label=lbl,color=c,lw=2.5)
ax.set_title("(A) Robot ON vs OFF  (N=40)")
ax.set_xlabel("Step"); ax.set_ylabel("% Evacuated")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
for ev, n, h, c in [(ev_h0,35,"h=0","#9090FF"),
                     (ev_h5,35,"h=5","#FF8800"),
                     (ev_h12,35,"h=12","#BB0000")]:
    s,f = evac_curve(ev,n); ax.plot(s,f,label=h,color=c,lw=2.5)
ax.set_title("(B) Prediction Horizon  (N=35)")
ax.set_xlabel("Step"); ax.set_ylabel("% Evacuated")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[2]
for ev, n, c in [(ev_20,20,"#55BB55"),(ev_40,40,"#EE8822"),(ev_70,70,"#CC2222")]:
    s,f = evac_curve(ev,n); ax.plot(s,f,label=f"N={n}",color=c,lw=2.5)
ax.set_title("(C) Agent Density")
ax.set_xlabel("Step"); ax.set_ylabel("% Evacuated")
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/content/experiments.png", dpi=150, bbox_inches="tight")
plt.show()
ipy_display(IPImage("/content/experiments.png"))
print("Saved /content/experiments.png")


## 12. Discussion

### Setup improvements

The arm is now mounted **at the centre** of the room (base at $(0,\,1,\,0)$),
giving symmetric reach to both exits ($\approx14.2$ units; arm reach $\approx15.5$).
Two interior pillar obstacles channel agents into lanes, creating clearer evacuation
streams and making the robot's blocking behaviour more visually obvious.

### Dual-exit targeting

`ExitCluster` tracks one centroid per exit independently. Each step,
`pick_robot_target` compares cluster sizes and sends the arm toward whichever
exit currently has more agents approaching. When the robot successfully scatters
one stream, that cluster shrinks and the other grows — triggering an automatic
switch. The console prints every switch event so the back-and-forth is quantifiable.

### Effect of prediction horizon (Experiment B)

| $T_h$ | Behaviour |
|---:|---|
| 0 | Purely reactive — arm chases current centroid, always arrives late |
| 5 | Leads the cluster; EE arrives near the exit as agents do |
| 12 | Over-prediction — arm swings past the exit opening entirely |

### Pillar obstacles

The pillars create two natural flow corridors (left and right of centre).
Without the robot, agents split roughly evenly between exits.
With the robot, blocking one corridor forces the entire stream through the other,
creating the exit-switching dynamic described in the assignment brief.

### Video Links

> **Main Simulation (N=40, Robot ON):** [YouTube link here]  
> **Comparison (Robot OFF):** [YouTube link here]

---

### References
- Vedo: https://vedo.embl.es/
- Helbing, D. et al. (2000). *Simulating dynamical features of escape panic.* Nature, 407, 487–490.
- Ribeiro, E. (2026). *robot_arms_inverse_kinematics.ipynb*. Florida Tech CSE5280.
